In [29]:
import os, shutil, pathlib, json, random
import numpy as np
import cv2
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

In [7]:
data_root = pathlib.Path("D:/FYP/dental-vision/ML/data/dentaldataset01/YOLO")

In [21]:
CLASS_NAMES = {
    0:  "Caries",
    6:  "Missing_Teeth",
    7:  "Periapical_Lesion",
    11: "Impacted_Tooth",
    13: "Bone_Loss",
}

In [22]:
OUTPUT = pathlib.Path("D:/FYP/dental-vision/ML/data/processed")

In [23]:
("Done.")

'Done.'

In [24]:
print(f"data_root: {data_root}")

data_root: D:\FYP\dental-vision\ML\data\dentaldataset01\YOLO


In [25]:
print(f"OUTPUT: {OUTPUT}")

OUTPUT: D:\FYP\dental-vision\ML\data\processed


In [26]:
def convert_split(split_name):
    img_dir   = data_root / split_name / "images"
    label_dir = data_root / split_name / "labels"

    copied  = 0
    skipped = 0

    for label_file in sorted(label_dir.glob("*.txt")):
        img_path = None
        for ext in [".jpg", ".jpeg", ".png"]:
            candidate = img_dir / (label_file.stem + ext)
            if candidate.exists():
                img_path = candidate
                break

        if img_path is None:
            skipped += 1
            continue

        seen_classes = set()
        for line in open(label_file).readlines():
            line = line.strip()
            if not line:
                continue
            cls_id = int(line.split()[0])
            if cls_id not in CLASS_NAMES:
                continue
            if cls_id in seen_classes:
                continue
            seen_classes.add(cls_id)

            dest_folder = OUTPUT / split_name / CLASS_NAMES[cls_id]
            dest_folder.mkdir(parents=True, exist_ok=True)
            dest = dest_folder / img_path.name
            shutil.copy2(img_path, dest)
            copied += 1

    print(f"{split_name:6s} → {copied} copies made, {skipped} skipped")

for split in ["train", "valid", "test"]:
    convert_split(split)

print("\nDone!")

train  → 14035 copies made, 0 skipped
valid  → 3970 copies made, 0 skipped
test   → 1994 copies made, 0 skipped

Done!


In [27]:
print(f"{'Disease':<22} {'train':>6} {'valid':>6} {'test':>6}")
print("-" * 44)

for cls_name in CLASS_NAMES.values():
    counts = []
    for split in ["train", "valid", "test"]:
        folder = OUTPUT / split / cls_name
        if folder.exists():
            counts.append(len(list(folder.glob("*.*"))))
        else:
            counts.append(0)
    print(f"{cls_name:<22} {counts[0]:>6} {counts[1]:>6} {counts[2]:>6}")

Disease                 train  valid   test
--------------------------------------------
Caries                   2182    614    269
Missing_Teeth            1230    257    173
Periapical_Lesion        1691    471    212
Impacted_Tooth           7655   2416   1340
Bone_Loss                1277    212      0


In [30]:
src = OUTPUT / "train" / "Bone_Loss"
dst = OUTPUT / "test"  / "Bone_Loss"
dst.mkdir(parents=True, exist_ok=True)

images = list(src.glob("*.*"))
random.seed(42)
to_move = random.sample(images, min(100, len(images)))

for img in to_move:
    shutil.move(str(img), dst / img.name)

print(f"Moved {len(to_move)} Bone Loss images from train to test.")

Moved 100 Bone Loss images from train to test.


In [31]:
print(f"{'Disease':<22} {'train':>6} {'valid':>6} {'test':>6}")
print("-" * 44)

for cls_name in CLASS_NAMES.values():
    counts = []
    for split in ["train", "valid", "test"]:
        folder = OUTPUT / split / cls_name
        if folder.exists():
            counts.append(len(list(folder.glob("*.*"))))
        else:
            counts.append(0)
    print(f"{cls_name:<22} {counts[0]:>6} {counts[1]:>6} {counts[2]:>6}")

Disease                 train  valid   test
--------------------------------------------
Caries                   2182    614    269
Missing_Teeth            1230    257    173
Periapical_Lesion        1691    471    212
Impacted_Tooth           7655   2416   1340
Bone_Loss                1177    212    100
